# Day 4 — Data Cleaning

This notebook practices basic data-cleaning techniques using a deliberately imperfect synthetic XR moral decision-making dataset.

The dataset contains missing values, duplicate observations, inconsistent category labels, invalid values, and an extreme observation.

The goal is to identify data-quality problems and create a clean version of the dataset before analysis.

In [56]:
import pandas as pd

df = pd.read_csv("/content/vr_moral_dilemma_day4_dirty.csv")

df.head()

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches
0,201,sheep,3.4,save_sheep,6.2,4.8,10
1,202,farmer,5.9,save_farmer,5.6,6.9,19
2,203,Sheep,4.7,save_farmer,6.8,6.1,16
3,204,farmr,2.8,save_sheep,4.7,3.9,7
4,205,sheep,NaN,save_sheep,6.0,5.2,12


In [57]:
df.shape

(11, 7)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   participant_id    11 non-null     int64  
 1   condition         11 non-null     object 
 2   decision_time     10 non-null     object 
 3   choice            11 non-null     object 
 4   embodiment_score  10 non-null     float64
 5   conflict_score    11 non-null     float64
 6   gaze_switches     11 non-null     int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 748.0+ bytes


In [59]:
df.isna().sum()

,0
participant_id,0
condition,0
decision_time,1
choice,0
embodiment_score,1
conflict_score,0
gaze_switches,0


In [60]:
df.duplicated().sum()

np.int64(1)

In [61]:
df["condition"].unique()

array(['sheep', 'farmer', 'Sheep', 'farmr'], dtype=object)

In [62]:
df["choice"].unique()

array(['save_sheep', 'save_farmer'], dtype=object)

In [63]:
df.dtypes

,0
participant_id,int64
condition,object
decision_time,object
choice,object
embodiment_score,float64
conflict_score,float64
gaze_switches,int64


In [104]:
df["decision_time"] = pd.to_numeric(
    df["decision_time"],
    errors="coerce"
)

df.dtypes

/tmp/ipykernel_4282/4089015782.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["decision_time"] = pd.to_numeric(


,0
participant_id,int64
condition,object
decision_time,float64
choice,object
embodiment_score,float64
conflict_score,float64
gaze_switches,int64


In [ ]:
df["condition"].unique()

In [68]:
df["condition"] = df["condition"].str.lower()

df["condition"] = df["condition"].replace({
    "farmr": "farmer"
})

In [69]:
df["condition"].unique()

array(['sheep', 'farmer'], dtype=object)

In [78]:
df[
    (df["embodiment_score"] < 1)
    | (df["embodiment_score"] > 7)
]

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches
7,208,farmer,4.2,save_farmer,8.3,5.7,14


In [79]:
df[
    (df["conflict_score"] < 1)
    | (df["conflict_score"] > 7)
]

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches
5,206,farmer,48.3,save_farmer,5.1,7.4,23


## Outlier Inspection

Extreme values are inspected before any exclusion decision is made.

In [80]:
df["decision_time"].describe()

,decision_time
count,8.000000
mean,9.750000
std,15.608148
min,2.800000
25%,3.550000
50%,4.450000
75%,5.300000
max,48.300000


In [84]:
df.sort_values("decision_time")

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches
3,204,farmer,2.8,save_sheep,4.7,3.9,7
0,201,sheep,3.4,save_sheep,6.2,4.8,10
9,210,farmer,3.6,save_sheep,4.8,2.9,8
7,208,farmer,4.2,save_farmer,8.3,5.7,14
2,203,sheep,4.7,save_farmer,6.8,6.1,16
6,207,sheep,5.1,save_sheep,NaN,6.6,18
1,202,farmer,5.9,save_farmer,5.6,6.9,19
5,206,farmer,48.3,save_farmer,5.1,7.4,23
4,205,sheep,NaN,save_sheep,6.0,5.2,12
8,209,sheep,NaN,save_sheep,5.9,4.4,11


In [83]:
df[df["decision_time"] > 20]

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches
5,206,farmer,48.3,save_farmer,5.1,7.4,23


An unusually high decision time (48.3 s) was identified for participant 206.
The value was not excluded solely because it was an outlier. However, this participant was later excluded because the same observation contained an invalid conflict score outside the valid 1–7 range.

In [85]:
clean_df = df.copy()

In [86]:
clean_df.duplicated().sum()

np.int64(0)

In [87]:
clean_df = clean_df.drop_duplicates()

In [88]:
clean_df.duplicated().sum()

np.int64(0)

In [89]:
clean_df.isna().sum()

,0
participant_id,0
condition,0
decision_time,2
choice,0
embodiment_score,1
conflict_score,0
gaze_switches,0


In [92]:
clean_df = clean_df.dropna()

## Handling Missing Values

Missing values are inspected before deciding how they should be handled.

In [93]:
clean_df.isna().sum()

,0
participant_id,0
condition,0
decision_time,0
choice,0
embodiment_score,0
conflict_score,0
gaze_switches,0


In [94]:
clean_df = clean_df[
    (clean_df["embodiment_score"].between(1, 7))
    & (clean_df["conflict_score"].between(1, 7))
]

In [95]:
clean_df[
    (clean_df["embodiment_score"] < 1)
    | (clean_df["embodiment_score"] > 7)
    | (clean_df["conflict_score"] < 1)
    | (clean_df["conflict_score"] > 7)
]

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches


In [96]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 0 to 9
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   participant_id    5 non-null      int64  
 1   condition         5 non-null      object 
 2   decision_time     5 non-null      float64
 3   choice            5 non-null      object 
 4   embodiment_score  5 non-null      float64
 5   conflict_score    5 non-null      float64
 6   gaze_switches     5 non-null      int64  
dtypes: float64(3), int64(2), object(2)
memory usage: 320.0+ bytes


In [97]:
clean_df.isna().sum()

,0
participant_id,0
condition,0
decision_time,0
choice,0
embodiment_score,0
conflict_score,0
gaze_switches,0


In [98]:
clean_df.duplicated().sum()

np.int64(0)

In [99]:
clean_df["condition"].unique()

array(['sheep', 'farmer'], dtype=object)

In [100]:
clean_df.to_csv(
    "vr_moral_dilemma_clean.csv",
    index=False
)

In [101]:
clean_df

,participant_id,condition,decision_time,choice,embodiment_score,conflict_score,gaze_switches
0,201,sheep,3.4,save_sheep,6.2,4.8,10
1,202,farmer,5.9,save_farmer,5.6,6.9,19
2,203,sheep,4.7,save_farmer,6.8,6.1,16
3,204,farmer,2.8,save_sheep,4.7,3.9,7
9,210,farmer,3.6,save_sheep,4.8,2.9,8


In [103]:
print("Raw dataset shape:", df.shape)
print("Clean dataset shape:", clean_df.shape)

Raw dataset shape: (10, 7)
Clean dataset shape: (5, 7)


## Data Cleaning Summary

The dataset was inspected for missing values, duplicate observations, inconsistent category labels, incorrect data types, invalid scale values, and extreme observations.

- `decision_time` was converted to a numeric datatype, with invalid text values coerced to missing values.
- Condition labels were standardized.
- Duplicate observations were removed.
- Rows containing missing values were excluded for this practice analysis.
- Scores outside the valid 1–7 range were excluded.
- An extreme decision time of 48.3 seconds was identified but was not considered invalid solely because it was an outlier. The corresponding participant was ultimately excluded because the same row contained an invalid conflict score.

The resulting cleaned dataset was saved separately from the raw dataset.